Cell 1: Load the master table and add exploratory metrics

In [26]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROCESSED = ROOT / "data" / "processed"
if not PROCESSED.exists():
    ROOT = Path(r"C:\Users\ASUS\ipl-business-intelligence")
    PROCESSED = ROOT / "data" / "processed"

master = pd.read_csv(PROCESSED / "player_season_master.csv")

# Exploratory performance measure (Phase 4 only; Phase 5 builds a proper score):
# a wicket is treated as worth about 20 runs.
master["points"] = master["runs"] + 20 * master["wickets"]
master["points_per_crore"] = master["points"] / master["price_cr"]
master["origin"] = master["is_overseas"].map({True: "Overseas", False: "Indian"}).fillna("Unknown")
master["season"] = master["season"].astype(int)

# players who played at least 5 matches: fairer for value comparisons
master["regular"] = master["matches_played"] >= 5

# The 2022 and 2024 files only contain auction purchases, while 2023 also has retained players.
# For a fair comparison across seasons we keep auction purchases only. Set to False to include all.
print("Acquisition types in the file:")
print(master["acquisition"].value_counts().to_string())
AUCTION_ONLY = True
if AUCTION_ONLY:
    master = master[master["acquisition"] == "sold"].copy()
    print("\nUsing auction purchases only:", len(master), "rows")

print("Rows:", len(master), "| columns:", len(master.columns))
master[["season", "player_name", "team", "role", "origin", "price_cr", "matches_played", "runs", "wickets", "points"]].head()

Acquisition types in the file:
acquisition
sold        356
retained    158

Using auction purchases only: 356 rows
Rows: 356 | columns: 26


,season,player_name,team,role,origin,price_cr,matches_played,runs,wickets,points
0,2022,Robin Uthappa,CSK,Batter,Indian,2.00,11.0,230.0,0.0,230.0
1,2022,Dwayne Bravo,CSK,All-Rounder,Overseas,4.40,10.0,23.0,16.0,343.0
2,2022,Ambati Rayudu,CSK,Wicketkeeper,Indian,6.75,11.0,274.0,0.0,274.0
3,2022,Deepak Chahar,CSK,Bowler,Indian,14.00,0.0,0.0,0.0,0.0
4,2022,C.Hari Nishaanth,CSK,Batter,Indian,0.20,0.0,0.0,0.0,0.0


Cell 2: What does the data cover?

In [27]:
overview = master.groupby("season").agg(
    players=("player_name", "count"),
    total_spend_cr=("price_cr", "sum"),
    median_price_cr=("price_cr", "median"),
    played=("played", "sum"),
    regulars=("regular", "sum"),
).round(1)
overview["played_pct"] = (overview["played"] / overview["players"] * 100).round(0)
print(overview.to_string())

print("\nPurchases by acquisition type:")
print(master.groupby(["season", "acquisition"]).size().to_string())
print("\nNationality known for:", (master["origin"] != "Unknown").mean().round(2) * 100, "% of rows")
print("Rows per role:")
print(master["role"].value_counts(dropna=False).to_string())

        players  total_spend_cr  median_price_cr  played  regulars  played_pct
season                                                                        
2022        204           551.7              1.0     145       104        71.0
2023         80           167.0              0.5      51        27        64.0
2024         72           230.4              1.2      46        30        64.0

Purchases by acquisition type:
season  acquisition
2022    sold           204
2023    sold            80
2024    sold            72

Nationality known for: 100.0 % of rows
Rows per role:
role
All-Rounder     135
Bowler          120
Batter           60
Wicketkeeper     41


Cell 3: How are prices distributed?

In [29]:
print(master["price_cr"].describe().round(2).to_string())
print("\nPurchases at 10 crore or more:", (master["price_cr"] >= 10).sum(),
      "| share of all rows:", f"{(master['price_cr'] >= 10).mean():.1%}")
print("Share of total spend that went to those purchases:",
      f"{master.loc[master['price_cr'] >= 10, 'price_cr'].sum() / master['price_cr'].sum():.1%}")

fig = px.histogram(master, x="price_cr", nbins=30, color=master["season"].astype(str),
                   title="Auction price distribution (crore)", labels={"price_cr": "Price (crore)", "color": "Season"})
fig.show()

count    356.00
mean       2.67
std        3.80
min        0.20
25%        0.20
50%        0.85
75%        3.80
max       24.75

Purchases at 10 crore or more: 22 | share of all rows: 6.2%
Share of total spend that went to those purchases: 31.6%


Cell 4: Does a higher price mean better performance?

In [30]:
played = master[master["played"]]

print("Spearman correlation between price and points (players who played):")
print("  All seasons:", round(played["price_cr"].corr(played["points"], method="spearman"), 2))
for s, g in played.groupby("season"):
    print(f"  {s}:", round(g["price_cr"].corr(g["points"], method="spearman"), 2))

print("\nBy role (players who played):")
for r, grp in played.groupby("role"):
    print(f"  {r:<14} n={len(grp):<4}", round(grp["price_cr"].corr(grp["points"], method="spearman"), 2))

fig = px.scatter(played, x="price_cr", y="points", color="role", hover_name="player_name",
                 hover_data=["season", "team", "matches_played", "runs", "wickets"],
                 title="Price vs performance (players who played)",
                 labels={"price_cr": "Price (crore)", "points": "Runs + 20 x wickets"})
fig.show()

Spearman correlation between price and points (players who played):
  All seasons: 0.52
  2022: 0.63
  2023: 0.43
  2024: 0.3

By role (players who played):
  All-Rounder    n=88   0.54
  Batter         n=44   0.6
  Bowler         n=85   0.38
  Wicketkeeper   n=25   0.56


Cell 5: What do the price bands buy?

In [31]:
# Bands include their upper value: (0,1], (1,3], (3,6], (6,10], (10,100]
bands = pd.cut(master["price_cr"], bins=[0, 1, 3, 6, 10, 100],
               labels=["up to 1 cr", "1-3 cr", "3-6 cr", "6-10 cr", "over 10 cr"])
band_tbl = master.groupby(bands, observed=True).agg(
    purchases=("player_name", "count"),
    played_pct=("played", lambda s: round(s.mean() * 100)),
    median_points=("points", "median"),
    total_points=("points", "sum"),
    total_spend_cr=("price_cr", "sum"),
)
band_tbl["points_per_crore"] = (band_tbl["total_points"] / band_tbl["total_spend_cr"]).round(1)
print(band_tbl.round(1).to_string())

fig = px.bar(band_tbl.reset_index(), x="price_cr", y="points_per_crore",
             title="Points per crore by price band", labels={"price_cr": "Price band", "points_per_crore": "Points per crore"})
fig.show()

            purchases  played_pct  median_points  total_points  total_spend_cr  points_per_crore
price_cr                                                                                        
up to 1 cr        194          55            1.5       11216.0            75.2             149.2
1-3 cr             67          76           65.0        7747.0           128.5              60.3
3-6 cr             37          84          149.0        6463.0           170.6              37.9
6-10 cr            40          92          294.0       11164.0           314.8              35.5
over 10 cr         18          94          409.5        6390.0           260.0              24.6


Cell 6: Best and worst value (players with 5+ matches)

In [32]:
cols = ["season", "player_name", "team", "role", "price_cr", "matches_played", "runs", "wickets", "points", "points_per_crore"]
regulars = master[master["regular"]]

print("BEST VALUE - most points per crore among players bought for 1 crore or more")
print("(20-lakh players always look cheap, so they are left out here):")
print(regulars[regulars["price_cr"] >= 1].sort_values("points_per_crore", ascending=False).head(15)[cols].round(1).to_string(index=False))

print("\nWORST VALUE - bought for 5 crore or more, fewest points per crore:")
print(regulars[regulars["price_cr"] >= 5].sort_values("points_per_crore").head(15)[cols].round(1).to_string(index=False))


BEST VALUE - most points per crore among players bought for 1 crore or more
(20-lakh players always look cheap, so they are left out here):
 season       player_name team         role  price_cr  matches_played  runs  wickets  points  points_per_crore
   2022      Devon Conway  CSK       Batter       1.0             7.0 252.0      0.0   252.0             252.0
   2022     Kuldeep Yadav   DC       Bowler       2.0            14.0  48.0     21.0   468.0             234.0
   2022    N. Tilak Varma   MI  All-Rounder       1.7            14.0 397.0      0.0   397.0             233.5
   2022       Tim Southee  KKR       Bowler       1.5             9.0   2.0     14.0   282.0             188.0
   2022       Umesh Yadav  KKR       Bowler       2.0            12.0  55.0     16.0   375.0             187.5
   2022   Wriddhiman Saha   GT Wicketkeeper       1.9            11.0 317.0      0.0   317.0             166.8
   2022      David Miller   GT       Batter       3.0            16.0 481.0      0.

Cell 7: Money spent on players who did not play

In [33]:
dead = master[~master["played"]]
print("Purchases who never played that season:", len(dead), "of", len(master))
print("Money spent on them (crore):", round(dead["price_cr"].sum(), 1),
      f"= {dead['price_cr'].sum() / master['price_cr'].sum():.1%} of total spend\n")

print("Per season:")
print(dead.groupby("season")["price_cr"].agg(["count", "sum"]).round(1).to_string())

print("\nHighest-priced players who did not play (check if injury before blaming the price):")
print(dead.sort_values("price_cr", ascending=False).head(10)[["season", "player_name", "team", "price_cr", "match_method"]].to_string(index=False))

Purchases who never played that season: 114 of 356
Money spent on them (crore): 116.5 = 12.3% of total spend

Per season:
        count   sum
season             
2022       59  55.1
2023       29  23.5
2024       26  37.9

Highest-priced players who did not play (check if injury before blaming the price):
 season        player_name team  price_cr    match_method
   2022      Deepak Chahar  CSK      14.0 surname+initial
   2022       Jofra Archer   MI       8.0 surname+initial
   2022          Mark Wood  LSG       7.5 surname+initial
   2024        Shivam Mavi  LSG       6.4           exact
   2023        Shivam Mavi   GT       6.0           exact
   2024 Dilshan Madushanka   MI       4.6        no_match
   2024       Chris Woakes PBKS       4.2 surname+initial
   2024        Harry Brook   DC       4.0 surname+initial
   2024         Robin Minz   GT       3.6        no_match
   2023         Will Jacks  RCB       3.2 surname+initial


Cell 8: Team efficiency

In [34]:
team_tbl = master.groupby("team").agg(
    players=("player_name", "count"),
    spend_cr=("price_cr", "sum"),
    points=("points", "sum"),
    played_pct=("played", lambda s: round(s.mean() * 100)),
)
team_tbl["points_per_crore"] = (team_tbl["points"] / team_tbl["spend_cr"]).round(1)
team_tbl = team_tbl.sort_values("points_per_crore", ascending=False)
print(team_tbl.round(1).to_string())
print("\nCaution: the 2022 file has no retained players, so team totals are incomplete and not directly comparable.")

fig = px.bar(team_tbl.reset_index(), x="team", y="points_per_crore", color="spend_cr",
             title="Team efficiency: points per crore of auction spend",
             labels={"points_per_crore": "Points per crore", "spend_cr": "Total spend (cr)"})
fig.show()

      players  spend_cr  points  played_pct  points_per_crore
team                                                         
MI         37      85.1  4929.0          76              57.9
DC         34      81.4  4414.0          76              54.2
LSG        34      91.0  4256.0          68              46.8
RR         35      85.2  3825.0          71              44.9
PBKS       37     113.5  5011.0          59              44.1
RCB        32      82.8  3645.0          62              44.0
SRH        39     134.4  5759.0          74              42.8
KKR        39      84.3  3419.0          62              40.6
CSK        34      94.4  3818.0          65              40.4
GT         35      97.0  3904.0          66              40.3

Caution: the 2022 file has no retained players, so team totals are incomplete and not directly comparable.


Cell 9: Indian vs overseas, by role, by season

In [35]:
def group_summary(df, by):
    t = df.groupby(by).agg(
        purchases=("player_name", "count"),
        median_price_cr=("price_cr", "median"),
        played_pct=("played", lambda s: round(s.mean() * 100)),
        median_points=("points", "median"),
        total_points=("points", "sum"),
        total_spend_cr=("price_cr", "sum"),
    )
    t["points_per_crore"] = (t["total_points"] / t["total_spend_cr"]).round(1)
    return t.round(1)

print("By nationality:")
print(group_summary(master, "origin").to_string())
print("\nBy role:")
print(group_summary(master, "role").to_string())
print("\nBy season:")
print(group_summary(master, "season").to_string())
print(group_summary(master, "season").to_string())

By nationality:
          purchases  median_price_cr  played_pct  median_points  total_points  total_spend_cr  points_per_crore
origin                                                                                                         
Indian          229              0.5          64           20.0       23710.0           435.4              54.5
Overseas        126              2.0          75          110.5       19270.0           513.6              37.5
Unknown           1              0.2           0            0.0           0.0             0.2               0.0

By role:
              purchases  median_price_cr  played_pct  median_points  total_points  total_spend_cr  points_per_crore
role                                                                                                               
All-Rounder         135              0.6          65           38.0       15539.0           368.0              42.2
Batter               60              1.4          73           62.

Cell 10: Indian vs overseas, like for like (same price band)

In [36]:
band3 = pd.cut(master["price_cr"], bins=[0, 1, 3, 100], labels=["up to 1 cr", "1-3 cr", "over 3 cr"])
like = master[master["origin"] != "Unknown"].groupby([band3, "origin"], observed=True).agg(
    purchases=("player_name", "count"),
    played_pct=("played", lambda s: round(s.mean() * 100)),
    total_points=("points", "sum"),
    total_spend_cr=("price_cr", "sum"),
)
like["avg_price_cr"] = (like["total_spend_cr"] / like["purchases"]).round(2)
like["points_per_player"] = (like["total_points"] / like["purchases"]).round(0)
like["points_per_crore"] = (like["total_points"] / like["total_spend_cr"]).round(1)
print(like[["purchases", "avg_price_cr", "points_per_player", "played_pct", "points_per_crore"]].to_string())

print("\nPlayers still with unknown nationality (should be none):")
print(master[master["origin"] == "Unknown"][["season", "player_name", "team", "price_cr"]].to_string(index=False))

                     purchases  avg_price_cr  points_per_player  played_pct  points_per_crore
price_cr   origin                                                                            
up to 1 cr Indian          153          0.31               54.0          52             172.0
           Overseas         40          0.68               74.0          68             110.1
1-3 cr     Indian           28          1.88              121.0          86              64.1
           Overseas         39          1.94              112.0          69              57.7
over 3 cr  Indian           48          6.98              252.0          92              36.1
           Overseas         47          8.74              254.0          87              29.0

Players still with unknown nationality (should be none):
 season      player_name team  price_cr
   2024 Swastik Chhikara   DC       0.2
